In [2]:
import requests
import pandas as pd
import os
import time
from dotenv import load_dotenv

In [14]:
load_dotenv()
github_key = os.getenv("GITHUB_TOKEN")

headers = {
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {github_key}"
}

urls: list[str] = []

urls.append("https://api.github.com/repos/nodejs/node/pulls?state=all")
urls.append("https://api.github.com/repos/angular/angular/pulls?state=all")
urls.append("https://api.github.com/repos/vuejs/core/pulls?state=all")
urls.append("https://api.github.com/repos/vercel/next.js/pulls?state=all")
urls.append("https://api.github.com/repos/facebook/react/pulls?state=all")
urls.append("https://api.github.com/repos/sveltejs/svelte/pulls?state=all")
urls.append("https://api.github.com/repos/withastro/astro/pulls?state=all")
urls.append("https://api.github.com/repos/QwikDev/qwik/pulls?state=all")
# response = requests.get(url, headers=headers)
# data = response.json()

In [32]:
max_pages = 125

def get_data(urls: list[str]) -> list[str]:
    all_results: list[str] = []
    for url in urls:
        page = 0
        print(f'Current url: {url}')
        while url:
            page += 1
            response = requests.get(url, headers=headers)

            if response.status_code == 200:
                # print(response.headers.get("Link", ""))
                print(f'Current page: {page}')
                data = response.json()
                
                all_results.extend(data)

                link_header = response.headers.get("Link", "")
                next_url = None
                for link in link_header.split(","):
                    if 'rel="next"' in link:
                        next_url = link[link.find("<")+1:link.find(">")]
                        break

                url = next_url

                # early return
                if page == max_pages:
                    url = None

                # 'don't get banned' check
                time.sleep(0.3)
                
            elif response.status_code == 202:
                print("Compiling data, try again shortly")
                break
            else:
                print(f"Error: {response.status_code}")
                break
    return all_results

In [33]:
all_results = get_data(urls)

Current url: https://api.github.com/repos/nodejs/node/pulls?state=all
Current page: 1
Current page: 2
Current page: 3
Current page: 4
Current page: 5
Current page: 6
Current page: 7
Current page: 8
Current page: 9
Current page: 10
Current page: 11
Current page: 12
Current page: 13
Current page: 14
Current page: 15
Current page: 16
Current page: 17
Current page: 18
Current page: 19
Current page: 20
Current page: 21
Current page: 22
Current page: 23
Current page: 24
Current page: 25
Current page: 26
Current page: 27
Current page: 28
Current page: 29
Current page: 30
Current page: 31
Current page: 32
Current page: 33
Current page: 34
Current page: 35
Current page: 36
Current page: 37
Current page: 38
Current page: 39
Current page: 40
Current page: 41
Current page: 42
Current page: 43
Current page: 44
Current page: 45
Current page: 46
Current page: 47
Current page: 48
Current page: 49
Current page: 50
Current page: 51
Current page: 52
Current page: 53
Current page: 54
Current page: 55
Curr

In [34]:
df = pd.json_normalize(
            all_results, 
            record_path=None, 
            meta=None, 
            errors='ignore'
        )
df.tail(100).to_json("../data/dataset/test/testo.json", orient="records")

In [35]:
cleaned_df = df.dropna(axis=1, how='all')
filtered_df = cleaned_df[["number", "state", "title", "body", "locked", "created_at", "updated_at", "closed_at", "merged_at", "assignees", "user.login", "labels", "author_association", "user.repos_url", "user.followers_url", "user.organizations_url", "user.starred_url", "user.type", "base.user.login", 'base.repo.name', "base.user.followers_url", "base.user.starred_url", "base.repo.created_at", "base.repo.updated_at", "base.repo.pushed_at", "base.repo.size", "base.repo.releases_url", "base.repo.stargazers_count", "base.repo.watchers_count", "base.repo.language", "base.repo.has_issues", "base.repo.has_projects", "base.repo.has_downloads", "base.repo.has_wiki", "base.repo.has_pages", "base.repo.has_discussions", "base.repo.forks_count"]]

In [36]:
non_url_columns_df = filtered_df.drop(columns=[col for col in filtered_df.columns if col.endswith('_url')])

In [37]:
non_url_columns_df.columns = ['number', 'state', 'title', 'body', 'locked', 'created_at', 'updated_at', 'closed_at', 'merged_at', 'assignees', 'user_name','labels','author_association','user_type','repo_owner_name', 'repo_name','repo_created_at','repo_updated_at','repo_pushed_at','repo_size','repo_stargazer_count','repo_watcher_count','repo_language', 'repo_has_issue', 'repo_has_projects', 'repo_has_downloads', 'repo_has_wiki', 'repo_has_pages', 'repo_has_discussions', 'repo_fork_count']

In [38]:
non_url_columns_df.to_csv("../data/dataset/pull_request_data.csv", index=False)
non_url_columns_df

,number,state,title,body,locked,created_at,updated_at,closed_at,merged_at,assignees,...,repo_stargazer_count,repo_watcher_count,repo_language,repo_has_issue,repo_has_projects,repo_has_downloads,repo_has_wiki,repo_has_pages,repo_has_discussions,repo_fork_count
0,58342,open,[v20.x] deps: V8: backport build fixes for Xco...,Node 20.19.2 currently fails to build from sou...,False,2025-05-15T05:09:16Z,2025-05-15T05:11:44Z,None,None,[],...,111257,111257,JavaScript,True,True,True,False,False,False,31553
1,58339,open,doc: add latest security release steward,As titled,False,2025-05-14T21:51:54Z,2025-05-15T04:53:38Z,None,None,[],...,111257,111257,JavaScript,True,True,True,False,False,False,31553
2,58337,open,lib: deprecate `_stream_*` modules,Runtime deprecation of all the `_stream_*` mod...,False,2025-05-14T18:38:39Z,2025-05-14T21:23:54Z,None,None,[],...,111257,111257,JavaScript,True,True,True,False,False,False,31553
3,58336,closed,Mock usdt,Mock usdt\r\n\r\n<!--\r\nBefore submitting a p...,False,2025-05-14T17:30:50Z,2025-05-14T17:59:06Z,2025-05-14T17:59:06Z,None,[],...,111257,111257,JavaScript,True,True,True,False,False,False,31553
4,58335,open,tools: add missing highway defines for IBM i,This was added for AIX but should have include...,False,2025-05-14T16:50:21Z,2025-05-14T20:34:08Z,None,None,[],...,111257,111257,JavaScript,True,True,True,False,False,False,31553
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,1833,closed,docs: update urls && minor changes,# What is it?\r\n\r\n- [ ] Feature / enhanceme...,False,2022-10-21T15:11:54Z,2022-10-21T18:45:18Z,2022-10-21T18:24:31Z,2022-10-21T18:24:31Z,[],...,21357,21357,TypeScript,True,True,True,False,False,False,1347
29996,1832,closed,chore: fixes,# What is it?\r\n\r\n- [ ] Feature / enhanceme...,False,2022-10-21T15:06:50Z,2022-10-21T15:38:24Z,2022-10-21T15:38:23Z,2022-10-21T15:38:23Z,[],...,21357,21357,TypeScript,True,True,True,False,False,False,1347
29997,1831,closed,docs(QwikCity): integrations react typo,# What is it?\r\n\r\n- [ ] Feature / enhanceme...,False,2022-10-21T14:22:02Z,2022-10-21T20:33:50Z,2022-10-21T15:04:00Z,2022-10-21T15:04:00Z,[],...,21357,21357,TypeScript,True,True,True,False,False,False,1347
29998,1830,closed,docs: Clarify React-specific snippet in React ...,# What is it?\r\n\r\n- [ ] Feature / enhanceme...,False,2022-10-21T13:40:42Z,2022-10-21T15:07:36Z,2022-10-21T15:07:35Z,2022-10-21T15:07:35Z,[],...,21357,21357,TypeScript,True,True,True,False,False,False,1347


url_columns = filtered_df.filter(regex='_url$')
url_columns

def fetch_url(url):
    try:
        print(url)
        response = requests.get(url)
        response.raise_for_status()
        return response.json()  # or response.text if it's not JSON
    except requests.RequestException as e:
        return f'Error: {e}'

# For each column, fetch and replace URLs with the data
count = 0
for col in url_columns.columns:
    print(f'col: {col}')
    count += 1
    url_columns[col] = url_columns[col].apply(fetch_url)
    time.sleep(1)
    if count == 30:
        break